# HW1 - Build the VAE step by step

Same workflow as in I2DL: you write the code in the `.py` files, this notebook checks each part and
lets you *see* what it does. Work top to bottom:

| Part | File | What you implement |
|---|---|---|
| 1 | `losses.py` | `kl_divergence` |
| 2 | `losses.py` | `reconstruction_nll`, `elbo_loss` |
| 3 | `models/vae.py` | `VAE.reparameterize` |
| 4 | `models/vae.py` | `Encoder`, `Decoder` |
| 5 | `models/vae.py` | `VAE.forward`, `VAE.sample`, `VAE.reconstruct` |
| 6 | - | overfit one image, then a small batch (sanity checks) |
| 7 | terminal | real training run, TensorBoard |

Sections 8 to 13 are the experiments for the report. Every run is saved under a time-stamped name and
`train_or_load` reuses the newest saved run of a name, so *Run All* only trains what is missing
(`RETRAIN = True` in section 7 trains everything again; `QUICK = True` trains 2 epochs).

`check("...")` runs the matching tests: **passed** = done, **skipped** = still a TODO, **failed** = look at the message.
Thanks to `autoreload`, a saved change in a `.py` file is used at once; no kernel restart needed.

The AI policy of the course (Piazza @35): the VAE architecture and the loss are yours. Everything in this notebook only *checks* your code.

In [ ]:
%load_ext autoreload
%autoreload 2

import math
import subprocess
import sys

import matplotlib.pyplot as plt
import torch
from torchvision.utils import make_grid

from dgm.data.chest_xray import ChestXray
from dgm.utils.misc import count_parameters, get_device, seed_everything
from dgm.utils.paths import REPO_ROOT
from hw1.config import ModelConfig
from hw1.losses import elbo_loss, kl_divergence
from hw1.models import VAE, Decoder, Encoder

seed_everything(0)
device = get_device()
SIZE = 64
print("device:", device)


def check(keyword):
    # Run the tests whose name contains `keyword` (same as: uv run pytest homeworks/hw1 -k keyword).
    result = subprocess.run(
        [
            sys.executable,
            "-m",
            "pytest",
            "homeworks/hw1/tests/test_vae.py",
            "-q",
            "-rs",
            "-k",
            keyword,
        ],
        capture_output=True,
        text=True,
        cwd=REPO_ROOT,
    )
    print(result.stdout[-1500:] or result.stderr[-1500:])


def show(images, title="", nrow=8):
    grid = make_grid(images.detach().cpu(), nrow=nrow, padding=2, pad_value=1.0)
    plt.figure(figsize=(nrow * 1.1, grid.shape[1] / grid.shape[2] * nrow * 1.1))
    plt.imshow(grid[0], cmap="gray", vmin=0, vmax=1)
    plt.title(title)
    plt.axis("off")
    plt.show()

## 0. One batch of data

The model sees tensors `x` of shape `[B, 1, 64, 64]` with values in `[0, 1]`. The labels are not used.

In [ ]:
train = ChestXray("train", SIZE)
x, labels = next(iter(torch.utils.data.DataLoader(train, batch_size=16, shuffle=True)))
print("x:", tuple(x.shape), x.dtype, f"values in [{x.min():.2f}, {x.max():.2f}]")
show(x, "16 training images")

## 1. The KL term - `kl_divergence` in `losses.py`

**On paper first.** For one latent dimension, $q = \mathcal N(\mu, \sigma^2)$ and $p = \mathcal N(0, 1)$:

$$\mathrm{KL}(q\,\|\,p) = \mathbb E_q\big[\log q(z) - \log p(z)\big] = \;?$$

Write the result with $\log\sigma^2$ (= `logvar`), then sum over the $K$ dimensions.

Questions for you (answer them in your own words before you code):
- Why is the KL zero exactly when $\mu = 0$ and $\sigma = 1$?
- The KL is measured in **nats** (natural log). What does "the KL of this image is 20 nats" mean in terms of information? (Hint: bits you would need to encode $z$ when you use the prior's code instead of the posterior's.)

In [ ]:
check("kl")

In [ ]:
# See it: KL of one dimension as a function of mu (sigma = 1) and of sigma (mu = 0).
try:
    mus = torch.linspace(-3, 3, 200)[:, None]
    sigmas = torch.linspace(0.05, 3, 200)[:, None]
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
    axes[0].plot(mus, kl_divergence(mus, torch.zeros_like(mus)))
    axes[0].set(xlabel="mu  (sigma = 1)", ylabel="KL [nats]")
    axes[1].plot(sigmas, kl_divergence(torch.zeros_like(sigmas), 2 * sigmas.log()))
    axes[1].set(xlabel="sigma  (mu = 0)", ylabel="KL [nats]")
    plt.tight_layout()
    plt.show()
except NotImplementedError as err:
    print("TODO:", err)

## 2. The reconstruction term and the ELBO - `reconstruction_nll`, `elbo_loss`

$-\log p_\theta(x \mid z)$, summed over the pixels. Two choices:
- **Bernoulli**: binary cross-entropy with logits. Our pixels are grayscale, not 0/1. Is that a problem for the *math*, for the *training*, or for neither? (Look at the pixel histogram in notebook 01.)
- **Gaussian** with fixed $\sigma$: a scaled squared error, plus a constant. Does the constant change the gradients? Does it change the reported ELBO?

Then `elbo_loss` puts the terms together. Why is `neg_elbo` the number to compare between runs, and not `loss`?

In [ ]:
check("nll or elbo")

## 3. The reparameterization trick - `VAE.reparameterize`

$z = \mu + \sigma \odot \varepsilon$ with $\varepsilon \sim \mathcal N(0, I)$. The randomness sits in $\varepsilon$, so the
gradient can flow back to $\mu$ and $\sigma$. The cell below draws many $z$ for $\mu = 2$, $\sigma = 0.5$:
the histogram must match the red density, and the gradients must exist.

In [ ]:
check("reparameterize")
try:
    mu = torch.full((20_000, 1), 2.0, requires_grad=True)
    logvar = torch.full((20_000, 1), math.log(0.25), requires_grad=True)
    z = VAE.reparameterize(mu, logvar)
    z.mean().backward()
    print(f"mean {z.mean():.3f} (want 2.0), std {z.std():.3f} (want 0.5)")
    print("gradient reaches mu:", mu.grad is not None, "| reaches logvar:", logvar.grad is not None)
    grid = torch.linspace(0, 4, 200)
    plt.hist(z.detach().flatten(), bins=80, density=True, alpha=0.6)
    plt.plot(grid, torch.distributions.Normal(2.0, 0.5).log_prob(grid).exp(), "r")
    plt.title("samples of z vs. N(2, 0.5^2)")
    plt.show()
except NotImplementedError as err:
    print("TODO:", err)

## 4. Encoder and decoder - `Encoder`, `Decoder`

Your architecture choices go into the report: layer types, depth, width (`hidden_channels`), how 64 x 64 shrinks
to a vector and grows back. Start simple; a few convolution layers are enough for a first run.

In [ ]:
check("encoder or decoder")
cfg = ModelConfig(latent_dim=32, hidden_channels=32)
try:
    encoder, decoder = Encoder(cfg, SIZE), Decoder(cfg, SIZE)
    mu, logvar = encoder(x)
    print(
        "encoder:",
        tuple(x.shape),
        "->",
        tuple(mu.shape),
        tuple(logvar.shape),
        f"| {count_parameters(encoder):,} parameters",
    )
    logits = decoder(mu)
    print(
        "decoder:",
        tuple(mu.shape),
        "->",
        tuple(logits.shape),
        f"| {count_parameters(decoder):,} parameters",
    )
except NotImplementedError as err:
    print("TODO:", err)

## 5. The whole model - `VAE.forward`, `VAE.sample`, `VAE.reconstruct`

- `forward`: encode, **sample** $z \sim q_\phi(z \mid x)$, decode. Used for training.
- `sample`: $z \sim p(z) = \mathcal N(0, I)$, decode. These are the **generated** images in the report.
- `reconstruct`: deterministic. Which $z$ do you use, and why?

An untrained model gives noise, of course. The point here is shapes and value ranges.

In [ ]:
check("forward or sample or reconstruct or gradients")
try:
    model = VAE(cfg, SIZE)
    out = model(x)
    print({name: tuple(t.shape) for name, t in vars(out).items()})
    show(model.sample(16), "samples of the UNTRAINED model")
except NotImplementedError as err:
    print("TODO:", err)

## 6. Sanity check: overfit on one single image

Take one image tensor from the data loader once, then reuse that exact image for every update.
A DataLoader provides batches; it is not itself a tensor and cannot be moved with `.to(device)`.
This check uses the likelihood and Gaussian sigma in `cfg`, with the standard KL weight of 1.
Reconstruction should improve, but stochastic latent sampling and KL regularization mean the VAE
objective does not require an exact pixel match. Compare the initial and final pixel MSE as well as the pictures.
With Gaussian likelihood, the fixed normalization constant can make the reconstruction NLL negative.

Question for you: with the Bernoulli likelihood, the reconstruction term stays in the thousands of nats even
when the reconstructions look perfect. Why can it not reach 0 for grayscale pixels? (Hint: what is the
smallest possible binary cross-entropy for a target pixel value of 0.5?)

In [ ]:
try:
    seed_everything(0)
    one_loader = torch.utils.data.DataLoader(train, batch_size=1, shuffle=True)
    x_one, _ = next(iter(one_loader))  # image tensor [1, 1, H, W], plus an unused label
    batch = x_one.to(device)  # selected once; never fetch a new image inside the loop
    model = VAE(cfg, SIZE).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    model.eval()
    initial_mse = (model.reconstruct(batch) - batch).square().mean().item()
    model.train()
    print(f"fixed batch: {tuple(batch.shape)} | likelihood: {cfg.likelihood}")
    history = []
    for _ in range(300):
        terms = elbo_loss(model(batch), batch, likelihood=cfg.likelihood, sigma=cfg.gaussian_sigma)
        if not torch.isfinite(terms["loss"]):
            raise FloatingPointError("One-image sanity check produced a non-finite loss")
        optimizer.zero_grad()
        terms["loss"].backward()
        optimizer.step()
        history.append({k: v.item() for k, v in terms.items()})
    plt.plot([h["recon"] for h in history], label="recon")
    plt.plot([h["kl"] for h in history], label="KL")
    plt.xlabel("step")
    plt.ylabel("nats per image")
    plt.legend()
    plt.show()
    print("last step:", {k: round(v, 1) for k, v in history[-1].items()})
    model.eval()
    reconstruction = model.reconstruct(batch)
    final_mse = (reconstruction - batch).square().mean().item()
    print(f"pixel MSE: {initial_mse:.6f} -> {final_mse:.6f}")
    fig, axes = plt.subplots(1, 2, figsize=(6, 3))
    for axis, image, title in zip(
        axes, [x_one[0, 0], reconstruction[0, 0].cpu()], ["Input", "Reconstruction"], strict=True
    ):
        axis.imshow(image.numpy(), cmap="gray", vmin=0, vmax=1)
        axis.set_title(title)
        axis.axis("off")
    plt.tight_layout()
    plt.show()
except NotImplementedError as err:
    print("TODO:", err)

## 6. 2nd Sanity check: overfit on a small batch size

Reuse the fixed batch `x` from section 0 and train a fresh model on those images.
This uses `cfg.likelihood` and `cfg.gaussian_sigma`, just like the single-image check.
Look for a falling loss and improved reconstructions. If learning stalls, inspect the data, gradients,
learning rate, and reconstruction/KL balance before interpreting it as an implementation error.

Question for you: with the Bernoulli likelihood, the reconstruction term stays in the thousands of nats even
when the reconstructions look perfect. Why can it not reach 0 for grayscale pixels? (Hint: what is the
smallest possible binary cross-entropy for a target pixel value of 0.5?)

In [ ]:
try:
    seed_everything(0)
    model = VAE(cfg, SIZE).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    batch = x.to(device)
    history = []
    for _ in range(300):
        terms = elbo_loss(model(batch), batch, likelihood=cfg.likelihood, sigma=cfg.gaussian_sigma)
        optimizer.zero_grad()
        terms["loss"].backward()
        optimizer.step()
        history.append({k: v.item() for k, v in terms.items()})
    plt.plot([h["recon"] for h in history], label="recon")
    plt.plot([h["kl"] for h in history], label="KL")
    plt.xlabel("step")
    plt.ylabel("nats per image")
    plt.legend()
    plt.show()
    print("last step:", {k: round(v, 1) for k, v in history[-1].items()})
    model.eval()
    show(
        torch.cat([x[:8], model.reconstruct(batch[:8]).cpu()]), "top: input, bottom: reconstruction"
    )
except NotImplementedError as err:
    print("TODO:", err)

## 7. Real training

Same pipeline as in I2DL: for every epoch, one pass over the training set (an update after every batch),
then one pass over the validation set (no updates). The losses go to **TensorBoard** after every epoch;
the model keeps the weights of the epoch with the lowest validation negative ELBO.

- **Settings:** the `hparams` dict below. The architecture keys are read by `models/vae.py`; their
  values below build the hand-written architecture exactly (the parameter count is printed at every start).
- **Quick check:** `QUICK = True` trains 2 epochs only (about 1 minute for the whole section), then set it back to `False`.
- **Speed:** one epoch is about 0.7 s on the laptop GPU for the base model, so 100 epochs are about 70 s.
- **Watch it:** in a terminal `uv run tensorboard --logdir runs/hw1-notebook`, then open <http://localhost:6006>.
- **Each run** returns `run["metrics"]`: validation MSE, PSNR, HF-MSE (MSE of the edges only, a 3x3 Laplacian
  filter: ribs and borders), and the number of active latent dimensions (mean KL > 0.01 nats).

In [ ]:
import copy
import csv
import itertools
import json
import pickle
from datetime import datetime

import numpy as np
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision.utils import save_image

from dgm.evaluation.metrics import InceptionScorer
from hw1.train import augment_batch

QUICK = False  # True: 2 epochs only, to see that everything runs
RETRAIN = False  # True: the experiment sections train again even when a saved run exists

hparams = {
    "name": "vae-base",
    # architecture (models/vae.py): these values build the hand-written architecture
    "latent_dim": 32,
    "hidden_channels": 32,
    "num_stages": 3,  # downsampling stages: 64 -> 32 -> 16 -> 8
    "convs_per_stage": 1,  # 3x3 conv layers per stage
    "norm": "batch",  # "batch", "group", or "none"
    "activation": "relu",  # "relu", "leaky_relu", or "silu"
    "downsample": "maxpool",  # "maxpool" or "stride" (learned 4x4 conv)
    "upsample": "transposed",  # "transposed" or "nearest_conv"
    "final_conv": False,  # extra 3x3 conv at full resolution before the logits
    "residual": False,  # x + F(x) around the extra convs of a stage (convs_per_stage > 1)
    "dropout": 0.0,  # dropout on the FC bottleneck (encoder features before the heads, decoder fc output)
    # likelihood (losses.py)
    "likelihood": "gaussian",  # "gaussian" (fixed sigma), "gaussian_calibrated" (sigma^2 = MSE), "bernoulli"
    "gaussian_sigma": 0.1,
    # training
    "epochs": 2 if QUICK else 50,
    "batch_size": 128,
    "lr": 1e-3,
    "lr_schedule": "none",  # "none" or "cosine" (anneal the lr to 0 over the run)
    "grad_clip_norm": 0.0,  # 0 = off; 1.0 caps the gradient norm (stability)
    "weight_decay": 0.0,  # decoupled weight decay (AdamW); 0 = plain Adam
    "augment": False,  # training images only: random shift <= 4 px, contrast +-10% (hw1.train.augment_batch)
    "beta": 1.0,  # weight of the KL term; 1.0 = the plain ELBO
}

val = ChestXray("val", SIZE)
RUNS_DIR = REPO_ROOT / "runs" / "hw1-notebook"  # TensorBoard logs
MODELS_DIR = REPO_ROOT / "homeworks" / "hw1" / "notebooks" / "models"  # pickled models
FIGURES_DIR = REPO_ROOT / "homeworks" / "hw1" / "report" / "figures"  # figures for the report
for folder in (RUNS_DIR, MODELS_DIR, FIGURES_DIR):
    folder.mkdir(parents=True, exist_ok=True)

In [ ]:
def build_vae(hparams):
    cfg = ModelConfig(
        latent_dim=hparams["latent_dim"],
        hidden_channels=hparams["hidden_channels"],
        likelihood=hparams["likelihood"],
        gaussian_sigma=hparams["gaussian_sigma"],
        # .get: older saved runs have no such keys; the defaults are the hand-written architecture
        num_stages=hparams.get("num_stages", 3),
        convs_per_stage=hparams.get("convs_per_stage", 1),
        norm=hparams.get("norm", "batch"),
        activation=hparams.get("activation", "relu"),
        downsample=hparams.get("downsample", "maxpool"),
        upsample=hparams.get("upsample", "transposed"),
        final_conv=hparams.get("final_conv", False),
        residual=hparams.get("residual", False),
        dropout=hparams.get("dropout", 0.0),
    )
    return VAE(cfg, SIZE).to(device)


def run_epoch(model, loader, hparams, optimizer=None):
    # One pass over the data. With an optimizer: training. Without: validation.
    # Returns the mean loss terms per image.
    totals = {"neg_elbo": 0.0, "recon": 0.0, "kl": 0.0}
    for x_batch, _ in loader:  # the labels are not used: the VAE models p(x)
        x_batch = x_batch.to(device)
        if optimizer is not None and hparams.get("augment", False):  # training images only
            x_batch = augment_batch(x_batch)
        terms = elbo_loss(
            model(x_batch),
            x_batch,
            beta=hparams["beta"],
            likelihood=hparams["likelihood"],
            sigma=hparams["gaussian_sigma"],
        )
        if not math.isfinite(terms["loss"].item()):
            raise RuntimeError(
                "Loss is NaN or Inf. Training stopped so later epochs do not all become NaN."
            )
        if optimizer is not None:
            optimizer.zero_grad()
            terms["loss"].backward()
            if hparams.get("grad_clip_norm", 0) > 0:  # cap the gradient norm (stability)
                torch.nn.utils.clip_grad_norm_(model.parameters(), hparams["grad_clip_norm"])
            optimizer.step()
        for name in totals:
            totals[name] += terms[name].item() * len(x_batch)
    return {name: value / len(loader.dataset) for name, value in totals.items()}


@torch.no_grad()
def reconstruct_all(model, dataset):
    # Posterior means and log-variances, reconstructions through mu, and the images of a dataset
    # (eval mode: no dropout, no augmentation).
    model.eval()
    mus, logvars, recons, images = [], [], [], []
    for x_batch, _ in DataLoader(dataset, batch_size=256):
        x_batch = x_batch.to(device)
        mu, logvar = model.encode(x_batch)
        mus.append(mu)
        logvars.append(logvar)
        recons.append(torch.sigmoid(model.decode(mu)))
        images.append(x_batch)
    return torch.cat(mus), torch.cat(logvars), torch.cat(recons), torch.cat(images)


@torch.no_grad()
def val_metrics(model):
    # Reconstruction quality through the posterior mean on the whole validation set: pixel MSE,
    # PSNR, the MSE of the edges only (3x3 Laplacian filter: ribs and borders), how many latent
    # dimensions carry information (mean KL over the images > 0.01 nats), and the same pixel MSE
    # on the training images, so that val_mse / train_mse - 1 is the overfitting gap.
    mu, logvar, recon, images = reconstruct_all(model, val)

    def edges(pictures):
        laplace = torch.tensor([[0.0, 1.0, 0.0], [1.0, -4.0, 1.0], [0.0, 1.0, 0.0]], device=device)
        return torch.nn.functional.conv2d(pictures, laplace.view(1, 1, 3, 3), padding=1)

    mse = ((recon - images) ** 2).mean().item()
    kl_per_dim = (-0.5 * (1 + logvar - mu**2 - logvar.exp())).mean(0)  # [K]
    _, _, train_recon, train_images = reconstruct_all(model, train)
    return {
        "val_mse": mse,
        "psnr": 10 * math.log10(1 / mse),
        "hf_mse": ((edges(recon) - edges(images)) ** 2).mean().item(),
        "active_dims": int((kl_per_dim > 0.01).sum()),
        "train_mse": ((train_recon - train_images) ** 2).mean().item(),
    }


def train_vae(hparams):
    # Train one VAE. Returns the model (weights of the best epoch) and a `run` dict with the
    # settings actually used, the loss history, the completed epochs, the best epoch, the
    # validation metrics of the best epoch (val_metrics), and the training time.
    seed_everything(hparams.get("seed", 0))  # default 0; section 14 trains three seeds
    model = build_vae(hparams)
    optimizer = torch.optim.AdamW(  # AdamW with weight_decay 0 is exactly Adam
        model.parameters(), lr=hparams["lr"], weight_decay=hparams.get("weight_decay", 0.0)
    )
    scheduler = None
    if hparams.get("lr_schedule", "none") == "cosine":  # anneal the lr to 0 over the run
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=hparams["epochs"])
    train_loader = DataLoader(train, batch_size=hparams["batch_size"], shuffle=True)
    val_loader = DataLoader(val, batch_size=hparams["batch_size"])
    run_name = f"{hparams['name']}-{datetime.now():%m%d-%H%M%S}"  # unique: never overwrites a run
    writer = SummaryWriter(RUNS_DIR / run_name)  # TensorBoard folder = run name = pickle name
    print(f"{run_name}: {count_parameters(model):,} parameters")
    history = {"train": [], "val": []}
    best_val, best_epoch, best_state = float("inf"), 0, None
    start = datetime.now()

    for epoch in range(1, hparams["epochs"] + 1):
        model.train()
        train_terms = run_epoch(model, train_loader, hparams, optimizer)
        model.eval()
        with torch.no_grad():
            val_terms = run_epoch(model, val_loader, hparams)

        history["train"].append(train_terms)
        history["val"].append(val_terms)
        for name in train_terms:
            writer.add_scalar(f"train/{name}", train_terms[name], epoch)
            writer.add_scalar(f"val/{name}", val_terms[name], epoch)
        writer.add_scalar("train/lr", optimizer.param_groups[0]["lr"], epoch)
        writer.add_images("samples", model.sample(16), epoch)
        if val_terms["neg_elbo"] < best_val:
            best_val, best_epoch = val_terms["neg_elbo"], epoch
            best_state = copy.deepcopy(model.state_dict())
        print(
            f"epoch {epoch:3d}/{hparams['epochs']}  train neg ELBO {train_terms['neg_elbo']:.1f}  "
            f"val neg ELBO {val_terms['neg_elbo']:.1f}  (recon {val_terms['recon']:.1f}, KL {val_terms['kl']:.1f})"
        )
        if scheduler is not None:
            scheduler.step()

    writer.close()
    if best_state is None:
        raise RuntimeError(
            "No epoch had a finite validation loss, so there is no checkpoint to load."
        )
    model.load_state_dict(best_state)
    run = {
        "name": run_name,
        "hparams": dict(hparams),
        "history": history,
        "epochs_completed": len(history["train"]),
        "best_epoch": best_epoch,
        "best_val_neg_elbo": best_val,
        "metrics": val_metrics(model),
        "train_time_s": round((datetime.now() - start).total_seconds(), 1),
    }
    print(f"best epoch {best_epoch}: {run['metrics']}  ({run['train_time_s']:.0f} s)")
    return model.eval(), run


def save_vae(model, run):
    # Like in I2DL: one pickle per run with the weights of the best epoch and the `run` dict.
    path = MODELS_DIR / f"{run['name']}.p"
    assert not path.exists(), f"{path.name} exists already; runs are never overwritten"
    state_dict = {name: tensor.cpu() for name, tensor in model.state_dict().items()}
    with open(path, "wb") as f:
        pickle.dump({**run, "state_dict": state_dict}, f)
    print(f"saved {path.name}")


def load_vae(name):
    # Returns the model in eval mode and its `run` dict. `name` is a saved run
    # (`<model name>-<date>-<time>`) or only the model name: then its newest saved run is used.
    # Pickles from before the `run` dict (only hparams + history) get the completed epochs and
    # best epoch from their history.
    path = MODELS_DIR / f"{name}.p"
    if not path.exists() and newest_run(name) is not None:
        path = MODELS_DIR / f"{newest_run(name)}.p"
    if not path.exists():
        saved_runs = sorted(file.stem for file in MODELS_DIR.glob("*.p"))
        raise FileNotFoundError(f"no saved run {name!r} in {MODELS_DIR}; saved runs: {saved_runs}")
    with open(path, "rb") as f:
        saved = pickle.load(f)
    model = build_vae(saved["hparams"])
    model.load_state_dict(saved["state_dict"])
    run = {key: value for key, value in saved.items() if key != "state_dict"}
    if "best_epoch" not in run:
        val_curve = [epoch["neg_elbo"] for epoch in run["history"]["val"]]
        run["name"] = path.stem
        run["epochs_completed"] = len(val_curve)
        run["best_epoch"] = val_curve.index(min(val_curve)) + 1
        run["best_val_neg_elbo"] = min(val_curve)
        run["old_format"] = True
        if run["hparams"]["epochs"] != run["epochs_completed"]:
            print(
                f"WARNING {name}: the saved setting says epochs={run['hparams']['epochs']}, "
                f"but the history has {run['epochs_completed']} epochs. Trust the history."
            )
    return model.eval(), run


def newest_run(name):
    # The newest saved run of `name` (every run is saved as `<name>-<date>-<time>.p`).
    files = sorted(MODELS_DIR.glob(f"{name}-[0-9]*.p"), key=lambda path: path.stat().st_mtime)
    return files[-1].stem if files else None


def train_or_load(name, settings):
    # Plumbing for the experiment sections: the newest saved run of `name` is reused, so Run All
    # only trains what is missing. RETRAIN = True (section 7) trains everything again.
    run_name = newest_run(name)
    if run_name is not None and not RETRAIN:
        print(f"=== {name}: saved run {run_name} (RETRAIN = True to train again) ===")
        return load_vae(run_name)
    print(f"\n=== {name} ===")
    model_i, run_i = train_vae(
        {**settings, "name": name, "epochs": 2 if QUICK else settings["epochs"]}
    )
    save_vae(model_i, run_i)
    return model_i, run_i

In [ ]:
model, run = train_vae(hparams)
save_vae(model, run)
print(
    f"run {run['name']}: best val neg ELBO {run['best_val_neg_elbo']:.1f} "
    f"at epoch {run['best_epoch']} of {run['epochs_completed']}"
)

### Look at the result

The same curves as in TensorBoard, as a matplotlib figure for the report (saved to `report/figures/`),
then new images (`z ~ N(0, I)`) and reconstructions of validation images (also saved).
A saved model comes back with `model, run = load_vae("<run name>")`; `run` holds the settings, the history, and the best epoch.

Questions for the report, with your numbers:
- How do the reconstruction term and the KL term change during training? Which one dominates, and why?
- Why are the samples blurrier than the reconstructions?
- What is the KL per image in nats at the end, and what does it say about how much information $z$ carries about $x$?

In [ ]:
def plot_history(history, name):
    epochs = range(1, len(history["train"]) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(12, 3))
    for ax, term in zip(axes, ["neg_elbo", "recon", "kl"], strict=True):
        ax.plot(epochs, [epoch[term] for epoch in history["train"]], label="train")
        ax.plot(epochs, [epoch[term] for epoch in history["val"]], "--", label="val")
        ax.set(title=term, xlabel="epoch", ylabel="nats per image")
        ax.legend()
    fig.suptitle(name)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / f"{name}_loss_curves.pdf")
    plt.show()


plot_history(run["history"], run["name"])

samples = model.sample(32)
x_val = next(iter(DataLoader(val, batch_size=8)))[0]
reconstructions = model.reconstruct(x_val.to(device)).cpu()
show(samples, "samples, z ~ N(0, I)")
show(torch.cat([x_val, reconstructions]), "val images (top) and reconstructions (bottom)")
save_image(samples, FIGURES_DIR / f"{run['name']}_samples.png", nrow=8, pad_value=1.0)
save_image(
    torch.cat([x_val, reconstructions]),
    FIGURES_DIR / f"{run['name']}_reconstructions.png",
    nrow=8,
    pad_value=1.0,
)

### FID and Inception Score

As in the course tutorial (PyTorch-Ignite): fresh samples against the 624 test images, 3 repeats -> mean and std.
`fid` is ignite's default (tiny values: it compares Inception class probabilities), `fid_2048` is the standard FID,
`inception_score` ignite's default IS. The "real images" row scores real training images the same way: the best
value a model can reach. About 2 minutes on the GPU. The scores are saved next to the model as JSON.

In [ ]:
test_images = ChestXray("test", SIZE).as_float()[: 128 if QUICK else None]
train_images = ChestXray("train", SIZE).as_float()[: len(test_images)]
scorer = InceptionScorer(device)


def fid_and_is(model, repeats=1 if QUICK else 3):
    # Scores of len(test_images) fresh samples against the test images: mean and std over the repeats.
    runs = [scorer(model.sample(len(test_images)).cpu(), test_images) for _ in range(repeats)]
    return {
        name: (np.mean([r[name] for r in runs]), np.std([r[name] for r in runs]))
        for name in runs[0]
    }


scores = fid_and_is(model)
real_scores = scorer(train_images, test_images)
for name, (mean, std) in scores.items():
    print(f"{name:16s} {mean:8.3f} +- {std:.3f}   (real images: {real_scores[name]:.3f})")
with open(MODELS_DIR / f"{run['name']}_scores.json", "w") as f:
    json.dump({"samples": scores, "real_images": real_scores}, f, indent=2)

## 8. Grid search

One training run per combination of the values in `GRID` (all other settings from `hparams`, fewer epochs).
Every run is saved as `models/grid-...-<date>-<time>.p` and reused on Run All; all curves appear together in
TensorBoard (`runs/hw1-notebook/grid-*`). Cost from scratch: runs x epochs x about 0.7 s (3 minutes for the nine).
At 20 epochs the runs are not converged (best epoch = last), so the grid ranks the latent size `K`; the
`beta` column is within noise at this length and is studied properly in section 13.

Compare the runs by the validation negative ELBO **only within one likelihood and sigma**: the Gaussian NLL has a
constant that depends on `gaussian_sigma`, and Bernoulli nats are a different quantity. Across likelihoods, use
`fid_and_is`. A different `beta` changes the training loss but not `neg_elbo`, so beta runs stay comparable.

In [ ]:
GRID = {
    "latent_dim": [32, 64, 128],
    "beta": [0.5, 1.0, 1.5],
}
grid_epochs = 20

results = []
for values in itertools.product(*GRID.values()):
    settings = dict(zip(GRID, values, strict=True))
    name = "grid-" + "-".join(f"{key}{value}" for key, value in settings.items())
    grid_model, grid_run = train_or_load(name, {**hparams, **settings, "epochs": grid_epochs})
    best_val = round(grid_run["best_val_neg_elbo"], 1)
    results.append({**settings, "val_neg_elbo": best_val, "name": grid_run["name"]})

### Compare the grid runs

The table (best first), the validation curves of all runs in one figure (for the report), and the best run's
samples. Put the best settings into `hparams` in section 7 and retrain with the full number of epochs:
that model is the one for the report.

In [ ]:
results.sort(key=lambda row: row["val_neg_elbo"])
for row in results:
    print(row)

plt.figure(figsize=(6, 3.5))
for row in results:
    _, grid_run = load_vae(row["name"])
    curve = [epoch["neg_elbo"] for epoch in grid_run["history"]["val"]]
    plt.plot(range(1, len(curve) + 1), curve, label=row["name"])
plt.xlabel("epoch")
plt.ylabel("val negative ELBO [nats per image]")
plt.legend(fontsize=7)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "grid_search.pdf")
plt.show()

best = results[0]
best_model, _ = load_vae(best["name"])
show(best_model.sample(16), f"best run: {best['name']}")
print("best settings for hparams:", {key: best[key] for key in GRID})

## 9. Compare saved models

Put the names of saved models (files in `notebooks/models/`, without `.p`) into `COMPARE`. Each model is loaded with
its own saved settings, in eval mode. Then:

- **Summary table:** latent size `K`, `beta`, completed epochs, best epoch (lowest validation negative ELBO), and the
  reconstruction and KL terms at that epoch. The table shows the ordinary negative ELBO (`recon + KL`, KL weight 1),
  not the beta-weighted training loss (`recon + beta * KL`), which is listed separately. Compare negative ELBOs only
  between runs with the same likelihood, `gaussian_sigma`, preprocessing, and data split; the table warns otherwise.
- **Reconstructions:** the first 8 validation images (always the same ones), one row per model, deterministic through the posterior mean.
- **New images:** 8 draws `z ~ N(0, I)` per model with a fixed seed (the global random state is restored afterwards,
  so later training is not affected). All 8 draws are shown, nothing is picked by hand. Columns of different models
  do not correspond to each other.

Both figures appear here and are saved as PNG and PDF in `report/figures/` with a time stamp, so earlier comparisons stay.
Pickles from before this cell (`old_format`) carry no metadata; their epoch counts and best epoch come from their history,
and a warning is printed when their saved `epochs` setting disagrees with it.

In [ ]:
COMPARE = [  # <-- names of saved models to compare (edit this list); the newest run of each name is used
    "vae-base",
    "vae-recipe",
    "a-stack",
]

models = {}  # name -> (model in eval mode, run dict)
for name in COMPARE:
    run_name = newest_run(name)
    if run_name is None:
        print(f"no saved run for {name!r}, skipped")
        continue
    models[name] = load_vae(run_name)

print(
    f"\n{'model':30s} {'K':>4} {'beta':>5} {'epochs':>7} {'best':>5} {'val neg ELBO':>13} "
    f"{'recon':>9} {'KL':>7} {'train loss(beta)':>17}  likelihood/sigma"
)
for name, (_, run) in models.items():
    hp, best = run["hparams"], run["best_epoch"]
    val_terms, train_terms = run["history"]["val"][best - 1], run["history"]["train"][best - 1]
    train_loss = train_terms["recon"] + hp["beta"] * train_terms["kl"]
    flag = " *" if run.get("old_format") else ""
    print(
        f"{name:30s} {hp['latent_dim']:4d} {hp['beta']:5.2f} {run['epochs_completed']:7d} {best:5d} "
        f"{val_terms['neg_elbo']:13.1f} {val_terms['recon']:9.1f} {val_terms['kl']:7.1f} {train_loss:17.1f}  "
        f"{hp['likelihood']}/{hp['gaussian_sigma']}{flag}"
    )
print("\n* = old pickle without metadata: epochs and best epoch taken from its history")
likelihoods = {
    (run["hparams"]["likelihood"], run["hparams"]["gaussian_sigma"]) for _, run in models.values()
}
if len(likelihoods) > 1:
    print(
        f"WARNING: different likelihood/sigma {sorted(likelihoods)}: do not compare the negative ELBOs"
    )

In [ ]:
x_fixed = next(iter(DataLoader(val, batch_size=8)))[
    0
]  # the first 8 validation images, always the same


def label_for(name, run):
    hp = run["hparams"]
    return f"{name}\nK={hp['latent_dim']}, beta={hp['beta']}"


def compare_figure(rows, title, filename):
    # One row of 8 grayscale images (values in [0, 1]) per entry of rows = [(label, images[8, 1, H, W]), ...]
    fig, axes = plt.subplots(len(rows), 8, figsize=(12, 1.6 * len(rows)), squeeze=False)
    for row_axes, (label, images) in zip(axes, rows, strict=True):
        for ax, image in zip(row_axes, images, strict=True):
            ax.imshow(image[0], cmap="gray", vmin=0, vmax=1)
            ax.set(xticks=[], yticks=[])
        row_axes[0].set_ylabel(label, rotation=0, ha="right", va="center", fontsize=7)
    fig.suptitle(title)
    fig.tight_layout()
    stamp = f"{datetime.now():%m%d-%H%M%S}"
    fig.savefig(FIGURES_DIR / f"{filename}_{stamp}.png", dpi=200)
    fig.savefig(FIGURES_DIR / f"{filename}_{stamp}.pdf")
    plt.show()


rows = [("validation\nimages", x_fixed)]
for name, (model_i, run_i) in models.items():
    rows.append((label_for(name, run_i), model_i.reconstruct(x_fixed.to(device)).cpu()))
compare_figure(rows, "reconstructions through the posterior mean", "compare_reconstructions")

In [ ]:
rows = []
for name, (model_i, run_i) in models.items():
    with (
        torch.random.fork_rng()
    ):  # fixed seed inside; the global random state is restored afterwards
        torch.manual_seed(0)
        rows.append((label_for(name, run_i), model_i.sample(8).cpu()))
compare_figure(rows, "new images, z ~ N(0, I), all 8 draws", "compare_samples")

### Optional: FID and Inception Score of the compared models

Expensive (about 2 minutes per model) and uses the **test set**, so only for the final evaluation. Run the FID cell
of section 7 first (it builds `scorer`, `test_images`, and `fid_and_is`), then set `COMPARE_FID = True`.

In [ ]:
COMPARE_FID = False
if COMPARE_FID:
    for name, (model_i, _) in models.items():
        print(
            name,
            {
                score: f"{mean:.3f} +- {std:.3f}"
                for score, (mean, std) in fid_and_is(model_i).items()
            },
        )

## 10. Experiments for the report

One table, one family of models (same data split, best epoch chosen on the validation set): the linear
baseline (PCA), the hand-written VAE as it was (`vae-base`), and then **one change at a time**, so that
every row answers one question (the ladder of `PLAN.md`):

- `vae-base`: the original architecture and recipe (50 epochs, constant lr, no clipping).
- `vae-recipe`: + longer training (200 epochs) with a cosine lr schedule and gradient clipping.
- `vae-deep`: + capacity (K = 128, two convs per stage, four stages, a final 3x3 conv).
- `vae-calibrated`: + the calibrated Gaussian likelihood (`sigma^2` = batch MSE) instead of `sigma = 0.1`.

`MODELS` holds the settings, `TRAIN` lists which of them to train now (each run is saved under its own
time-stamped name, nothing is overwritten). The parameter count printed at the start of every run shows
what was really built. Negative ELBOs are comparable only within one likelihood and sigma; for the
calibrated row compare MSE, HF-MSE, and FID instead.

In [ ]:
BASE = {  # the hand-written model and recipe, exactly as in section 7
    "latent_dim": 32,
    "hidden_channels": 32,
    "num_stages": 3,
    "convs_per_stage": 1,
    "norm": "batch",
    "activation": "relu",
    "downsample": "maxpool",
    "upsample": "transposed",
    "final_conv": False,
    "residual": False,
    "dropout": 0.0,
    "likelihood": "gaussian",
    "gaussian_sigma": 0.1,
    "epochs": 50,
    "batch_size": 128,
    "lr": 1e-3,
    "lr_schedule": "none",
    "grad_clip_norm": 0.0,
    "weight_decay": 0.0,
    "augment": False,
    "beta": 1.0,
}
MODELS = {"vae-base": BASE}
MODELS["vae-recipe"] = {**BASE, "epochs": 200, "lr_schedule": "cosine", "grad_clip_norm": 1.0}
MODELS["vae-deep"] = {
    **MODELS["vae-recipe"],
    "latent_dim": 128,
    "num_stages": 4,
    "convs_per_stage": 2,
    "final_conv": True,
}
MODELS["vae-calibrated"] = {**MODELS["vae-deep"], "likelihood": "gaussian_calibrated"}
TRAIN = list(MODELS)  # <-- which models to train now (saved runs are reused; all four from scratch: ~25 min)

for name in TRAIN:
    model_i, run_i = train_or_load(name, MODELS[name])

### Linear baseline: PCA

PCA with K components is the best linear model with a K-dimensional latent space; probabilistic PCA is
exactly a VAE with a linear encoder and decoder. So the PCA rows show what the convolutional VAE buys over a
linear one at the same K. The mean image is the K = 0 row. New images come from Gaussian scores with the
training variance of each component (that is probabilistic PCA's prior), decoded through the same components.

In [ ]:
PCA_KS = [32, 128, 256]
val_images = val.as_float()  # [476, 1, 64, 64] in [0, 1]
X_train = ChestXray("train", SIZE).as_float().flatten(1).to(device)
X_val = val_images.flatten(1).to(device)
mean_image = X_train.mean(0)
_, singular_values, components = torch.linalg.svd(X_train - mean_image, full_matrices=False)


def pca_reconstruct(x, k):
    basis = components[:k]
    return mean_image + (x - mean_image) @ basis.T @ basis


def pca_sample(k, n):
    # probabilistic PCA: Gaussian scores with the training variance of each component, decoded linearly
    std = singular_values[:k] / math.sqrt(len(X_train) - 1)
    scores = torch.randn(n, k, device=device) * std
    return (mean_image + scores @ components[:k]).clamp(0, 1).view(n, 1, SIZE, SIZE)


baseline_rows = [
    {
        "model": "mean image",
        "K": 0,
        "params": 0,
        "val_mse": ((X_val - mean_image) ** 2).mean().item(),
    }
]
for k in PCA_KS:
    mse = ((X_val - pca_reconstruct(X_val, k)) ** 2).mean().item()
    baseline_rows.append({"model": f"PCA K={k}", "K": k, "params": k * SIZE * SIZE, "val_mse": mse})
for row in baseline_rows:
    print(row)
with torch.random.fork_rng():
    torch.manual_seed(0)
    show(
        pca_sample(PCA_KS[-1], 16).cpu(), f"PCA K={PCA_KS[-1]}: new images from the fitted Gaussian"
    )

### Results table

One row per baseline and per saved VAE run (`RESULTS` names a model; the newest saved run with that name is used).
Columns: parameters, the validation metrics of `run["metrics"]` (pixel MSE, PSNR, HF-MSE of the edges, active
latent dimensions; the PCA rows have the MSE), and the validation negative ELBO with its reconstruction and
KL terms at the best epoch. FID-2048 and IS are added when a run has saved scores (the FID cell of section 7)
or when `SCORE_RESULTS = True` (test set, about 2 minutes per row). The table is printed and saved as CSV in
`report/figures/`.

In [ ]:
RESULTS = [  # <-- models to list: the ladder from the hand-written model to the final one
    "vae-base",
    "vae-recipe",
    "vae-deep",
    "vae-calibrated",
    "a-base",
    "a-stack",
    "r-stack",
]
SCORE_RESULTS = (
    False  # True: FID / IS for rows without saved scores (needs the FID cell of section 7)
)


def val_mse(model):
    # Only for pickles from before run["metrics"] existed.
    chunks = [
        model.reconstruct(val_images[i : i + 128].to(device)).cpu()
        for i in range(0, len(val_images), 128)
    ]
    return ((torch.cat(chunks) - val_images) ** 2).mean().item()


def add_scores(row, sampler):
    # FID-2048 and IS of len(test_images) new images against the test set (needs `scorer` from section 7)
    scores = scorer(sampler(len(test_images)).cpu(), test_images)
    row["fid_2048"], row["inception_score"] = scores["fid_2048"], scores["inception_score"]


rows = [dict(row) for row in baseline_rows]
if SCORE_RESULTS:
    for row in rows[1:]:  # PCA rows (the mean image has no spread, its FID is meaningless)
        add_scores(row, lambda n, k=row["K"]: pca_sample(k, n))
for name in RESULTS:
    run_name = newest_run(name)
    if run_name is None:
        print(f"no saved run for {name!r}, skipped")
        continue
    model_i, run_i = load_vae(run_name)
    hp, best = run_i["hparams"], run_i["history"]["val"][run_i["best_epoch"] - 1]
    metrics = run_i.get("metrics") or {"val_mse": val_mse(model_i)}  # older pickles: MSE only
    row = {
        "model": run_name,
        "K": hp["latent_dim"],
        "params": count_parameters(model_i),
        **metrics,
        "val_neg_elbo": best["neg_elbo"],
        "recon": best["recon"],
        "kl": best["kl"],
        "best_epoch": run_i["best_epoch"],
        "epochs": run_i["epochs_completed"],
        "likelihood": hp["likelihood"],
    }
    scores_file = MODELS_DIR / f"{run_name}_scores.json"
    if scores_file.exists():  # 3-repeat scores from the FID cell of section 7
        with open(scores_file) as f:
            saved_scores = json.load(f)["samples"]
        row["fid_2048"], row["inception_score"] = (
            saved_scores["fid_2048"][0],
            saved_scores["inception_score"][0],
        )
    elif SCORE_RESULTS:
        add_scores(row, model_i.sample)
    rows.append(row)

columns = list(dict.fromkeys(key for row in rows for key in row))
table = [
    [f"{row[c]:.4g}" if isinstance(row.get(c), float) else str(row.get(c, "-")) for c in columns]
    for row in rows
]
widths = [max(len(c), *(len(line[i]) for line in table)) for i, c in enumerate(columns)]
print("  ".join(c.rjust(w) for c, w in zip(columns, widths, strict=True)))
for line in table:
    print("  ".join(cell.rjust(w) for cell, w in zip(line, widths, strict=True)))

with open(FIGURES_DIR / f"results_table_{datetime.now():%m%d-%H%M%S}.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=columns)
    writer.writeheader()
    writer.writerows(rows)

## 11. A: one change at a time (architecture)

Which architecture change buys reconstruction quality? Fixed base: the hand-written `BASE` of section 10 (3 stages,
1 conv per stage, batch norm, ReLU, max-pool, transposed convs, no final conv, 32 channels) with K = 128,
`beta = 0.1` (close to a plain autoencoder, so the KL term does not hide capacity differences), the Gaussian
likelihood with `sigma = 0.1` (so `val_mse` and `hf_mse` stay comparable), 200 epochs, lr 1e-3 with a cosine
schedule, gradient clipping 1.0, batch 128. Every row is exactly one override. Target: the PCA row at the same K
(val MSE 0.00188 at K = 128).

Hypotheses written before the runs (claims, not facts):

1. More downsampling stages shrink the decoder's starting grid (`num_stages=4` starts at 4x4). That is not the way to get the ribs back.
2. The missing capacity is extra 3x3 convolutions at the resolutions that already exist, plus one 3x3 at full resolution.
3. Those extra convolutions need a residual add (`x + F(x)`) once they get deeper. At exactly two convolutions per stage the skip may tie the plain stack; a tie is an acceptable result.
4. A wider first channel count helps; SiLU is a small effect.

Runs are saved as `a-<change>-<stamp>.p` and reused on Run All. The `e1-<change>` pickles of 27 Oct are a second run
of the same settings on another kernel (same seed; cuDNN is not deterministic): the table lists them as `repeat`, which
shows the run-to-run noise of one setting.

In [ ]:
A_BASE = {
    **BASE,  # 3 stages, 1 conv per stage, batch norm, relu, maxpool, transposed, no final conv, 32 channels
    "latent_dim": 128,
    "beta": 0.1,
    "likelihood": "gaussian",
    "gaussian_sigma": 0.1,
    "epochs": 200,
    "lr": 1e-3,
    "lr_schedule": "cosine",
    "grad_clip_norm": 1.0,
    "batch_size": 128,
}
A = {  # name -> the single override on A_BASE
    "a-base": {},
    "a-stages4": {"num_stages": 4},
    "a-convs2": {"convs_per_stage": 2},
    "a-convs2-res": {"convs_per_stage": 2, "residual": True},
    "a-final": {"final_conv": True},
    "a-width64": {"hidden_channels": 64},
    "a-stride": {"downsample": "stride"},
    "a-nearest": {"upsample": "nearest_conv"},
    "a-silu": {"activation": "silu"},
}
A_TRAIN = list(A)  # <-- which rows to train now (saved runs are reused; all nine from scratch: ~45 min)

for name in A_TRAIN:
    model_i, run_i = train_or_load(name, {**A_BASE, **A[name]})

In [ ]:
A_REPEATS = {  # the e1-* runs of 27 Oct: same settings, second run (no twin for a-convs2-res and a-silu)
    "a-base": "e1-base",
    "a-stages4": "e1-stages4",
    "a-convs2": "e1-convs2",
    "a-final": "e1-finalconv",
    "a-width64": "e1-width64",
    "a-stride": "e1-stride",
    "a-nearest": "e1-nearest",
}
print("Reference (same validation split): PCA K=128 val_mse 0.00188 (train 0.00169), hf_mse 0.00853;")
print("                                   PCA K=256 val_mse 0.00118 (train 0.00097), hf_mse 0.00764")
print()
print(
    f"{'name':14s} {'params':>10s} {'best':>5s} {'val_mse':>8s} {'repeat':>8s} {'hf_mse':>8s} "
    f"{'psnr':>6s} {'train_mse':>10s} {'gap%':>5s} {'time s':>7s}"
)
for name in A:
    run_name = newest_run(name)
    if run_name is None:
        print(f"{name:14s} (no saved run)")
        continue
    model_i, run_i = load_vae(run_name)
    m = run_i["metrics"]
    repeat = newest_run(A_REPEATS.get(name, "-"))
    repeat_mse = load_vae(repeat)[1]["metrics"]["val_mse"] if repeat else float("nan")
    gap = 100 * (m["val_mse"] / m["train_mse"] - 1)
    print(
        f"{name:14s} {count_parameters(model_i):10,d} {run_i['best_epoch']:5d} {m['val_mse']:8.5f} "
        f"{repeat_mse:8.5f} {m['hf_mse']:8.5f} {m['psnr']:6.2f} {m['train_mse']:10.5f} {gap:5.0f} "
        f"{run_i.get('train_time_s', float('nan')):7.0f}"
    )

In [ ]:
WINNERS = ["a-stride", "a-silu", "a-nearest"]  # <-- from the table of 27 Oct by the rule below; edit if you disagree
# Decision rule (written before the runs; applied by hand to the table above):
# - Keep a row if its val_mse is at least 5% lower than a-base, or if its hf_mse is lower and
#   its val_mse is not more than 2% higher.
# - Keep num_stages=4 (a-stages4) only if it wins both val_mse and hf_mse against a-base.
#   Otherwise the stack stays at 3 stages.
# - Include a-convs2-res instead of a-convs2 only when its val_mse is lower. A tie keeps the
#   plain stack (a-convs2).
# - Include a-silu only when its val_mse is at least 5% lower than a-base.
# - Do not add gaussian_calibrated in this round.
# 27 Oct table: a-stride -19% val_mse (0.00180), a-silu -14% (0.00191), a-nearest -3% with lower hf_mse ->
# in; a-final lower hf_mse but +3% val_mse -> out; stages4, convs2, convs2-res, width64 -> worse.

stack_overrides = {}
for name in WINNERS:
    stack_overrides.update(A[name])
A_STACK = {**A_BASE, **stack_overrides}
print("a-stack overrides:", stack_overrides or "(none yet: fill WINNERS after the table)")
if WINNERS:
    model_i, run_i = train_or_load("a-stack", A_STACK)
    print({k: round(v, 5) if isinstance(v, float) else v for k, v in run_i["metrics"].items()})

## 12. R: regularization, one change at a time

The A runs show that the capacity is there: at epoch 200 the training MSE is between PCA K=128 and K=256, while
the validation MSE stops improving around epoch 90 and then rises (gap val / train - 1: about 50% for a-base and
a-stride, 65% for a-silu; PCA K=128 itself has 12%, K=256 22%). The next lever is generalization, not size.
Base: `a-stack` from section 11 (its winners on the e1-base recipe: K = 128, `beta = 0.1`, Gaussian `sigma = 0.1`,
200 epochs, lr 1e-3 cosine, clipping 1.0). `r-base` trains the same setting once more: the run-to-run noise.
One change per row:

- `augment`: training images only, random shift up to 4 px (edge pixels repeated) and contrast +-10%; no flips.
- `weight_decay`: decoupled (AdamW), two strengths. Note: it also decays the batch-norm scales and the biases.
- `dropout`: on the FC bottleneck (encoder features before the heads, decoder `fc` output), two strengths.

The table reports the validation MSE and the training MSE measured the same way (eval mode, posterior mean, no
augmentation), so `gap` = val / train - 1 is the overfitting measure. Runs are saved as `r-<change>-<stamp>.p` and
reused on Run All.

In [ ]:
R_BASE = A_STACK  # = a-stack: the architecture winners of section 11 on the e1-base recipe
R = {  # name -> the single change on R_BASE
    "r-base": {},
    "r-aug": {"augment": True},
    "r-wd0.01": {"weight_decay": 0.01},
    "r-wd0.1": {"weight_decay": 0.1},
    "r-drop0.2": {"dropout": 0.2},
    "r-drop0.5": {"dropout": 0.5},
}
R_TRAIN = list(R)  # <-- which rows to train now (saved runs are reused; all six from scratch: ~30 min)

for name in R_TRAIN:
    model_i, run_i = train_or_load(name, {**R_BASE, **R[name]})

In [ ]:
print("Reference (same validation split): PCA K=128 val_mse 0.00188 (train 0.00169), hf_mse 0.00853;")
print("                                   PCA K=256 val_mse 0.00118 (train 0.00097), hf_mse 0.00764")
print()
print(
    f"{'name':12s} {'params':>10s} {'best':>5s} {'val_mse':>9s} {'hf_mse':>9s} {'psnr':>6s} "
    f"{'train_mse':>10s} {'gap%':>6s} {'time s':>7s}"
)
for name in ["a-stack", *R]:  # a-stack = the base; r-base repeats it
    run_name = newest_run(name)
    if run_name is None:
        print(f"{name:12s} (no saved run)")
        continue
    model_i, run_i = load_vae(run_name)
    m = run_i["metrics"]
    gap = 100 * (m["val_mse"] / m["train_mse"] - 1)
    print(
        f"{name:12s} {count_parameters(model_i):10,d} {run_i['best_epoch']:5d} {m['val_mse']:9.5f} "
        f"{m['hf_mse']:9.5f} {m['psnr']:6.2f} {m['train_mse']:10.5f} {gap:6.1f} "
        f"{run_i.get('train_time_s', float('nan')):7.0f}"
    )

In [ ]:
R_WINNERS = []  # <-- fill in from the table above, e.g. ["r-aug", "r-drop0.2"]; empty until then
# Decision rule (as in section 11): keep a row if its val_mse is at least 5% lower than r-base, or if
# its hf_mse is lower and its val_mse is not more than 2% higher. Of two strengths of one knob keep
# only the better one. The gap column says whether a knob worked as a regularizer at all.

r_overrides = {}
for name in R_WINNERS:
    r_overrides.update(R[name])
R_STACK = {**R_BASE, **r_overrides}
print("r-stack overrides:", r_overrides or "(none yet: fill R_WINNERS after the table)")
if R_WINNERS:
    model_i, run_i = train_or_load("r-stack", R_STACK)
    print({k: round(v, 5) if isinstance(v, float) else v for k, v in run_i["metrics"].items()})

## 13. B: rate-distortion and sample quality (FID)

The handout's metric is the FID / IS of **samples** $z \sim N(0, I)$, not the reconstruction MSE. This section
puts the best setting so far on the rate-distortion curve and scores every point:

- Base: `r-stack` when section 12 has winners, otherwise `a-stack`. One run per KL weight `beta` in
  {0.03, 0.1, 0.3, 1} at `sigma = 0.1`, plus the calibrated likelihood (`sigma^2` = batch MSE) at `beta = 1`.
  Lower `beta`: lower distortion (val MSE), higher rate (KL nats per image). `b-beta0.1` repeats the base setting.
- Scores: FID-2048 and IS of `len(test_images)` new images against the test images, 3 repeats, saved to
  `models/<run>_scores.json` exactly like section 7 (needs `scorer` from the FID cell of section 7 and
  `baseline_rows` / `pca_sample` from the PCA cell of section 10). The PPCA sampler is the linear generative baseline.
- Figure: val MSE against KL with the PCA lines, FID written at every point (`report/figures/rate_distortion_<stamp>.pdf`).
- Aggregate posterior: samples from the prior land in regions of the latent space that no training image is encoded
  to. The last cell fits one Gaussian to the encoded training set (the aggregate posterior, moment-matched) and scores
  samples drawn from it instead of from $N(0, I)$ (Dai & Wipf 2019). Same decoder, different $z$; the FID difference
  is the size of that mismatch.

In [ ]:
B_BASE = R_STACK  # r-stack when section 12 has winners, otherwise a-stack
B = {  # name -> the change on B_BASE: the KL weight at sigma = 0.1, plus the calibrated likelihood
    "b-beta0.03": {"beta": 0.03},
    "b-beta0.1": {"beta": 0.1},
    "b-beta0.3": {"beta": 0.3},
    "b-beta1": {"beta": 1.0},
    "b-calibrated": {"beta": 1.0, "likelihood": "gaussian_calibrated"},
}
B_TRAIN = list(B)  # <-- which points to train now (saved runs are reused; all five from scratch: ~25 min)
RESCORE = False  # True: compute FID / IS again even when a `_scores.json` exists

b_models = {}  # name -> (model, run)
for name in B_TRAIN:
    b_models[name] = train_or_load(name, {**B_BASE, **B[name]})

# FID-2048 and IS of len(test_images) new images against the test images, 3 repeats, like section 7
for name, (model_i, run_i) in b_models.items():
    scores_file = MODELS_DIR / f"{run_i['name']}_scores.json"
    if scores_file.exists() and not RESCORE:
        continue
    scores = fid_and_is(model_i)
    with open(scores_file, "w") as f:
        json.dump({"samples": scores, "real_images": real_scores}, f, indent=2)
    print(name, {score: f"{mean:.3f} +- {std:.3f}" for score, (mean, std) in scores.items()})

In [ ]:
def load_scores(run_name, key="samples"):
    # (mean, std) per score from `models/<run>_scores.json`, or None when the run was not scored yet
    scores_file = MODELS_DIR / f"{run_name}_scores.json"
    if not scores_file.exists():
        return None
    with open(scores_file) as f:
        return json.load(f).get(key)


pca_lines = {row["model"]: row["val_mse"] for row in baseline_rows if row["K"] in (128, 256)}
ppca_scores = scorer(pca_sample(128, len(test_images)).cpu(), test_images)  # linear generative baseline
print(
    f"PPCA K=128 sampler: FID-2048 {ppca_scores['fid_2048']:.1f}, IS {ppca_scores['inception_score']:.2f};  "
    f"real images: FID-2048 {real_scores['fid_2048']:.1f}, IS {real_scores['inception_score']:.2f}"
)
print(
    f"{'name':14s} {'beta':>5s} {'sigma':>6s} {'best':>5s} {'KL':>7s} {'val_mse':>8s} {'hf_mse':>8s} "
    f"{'act':>4s} {'FID-2048':>14s} {'IS':>13s}"
)
points = []  # (name, KL, val MSE, FID) for the figure
for name in B:
    run_name = newest_run(name)
    if run_name is None:
        print(f"{name:14s} (no saved run)")
        continue
    _, run_i = load_vae(run_name)
    hp, m = run_i["hparams"], run_i["metrics"]
    best = run_i["history"]["val"][run_i["best_epoch"] - 1]
    scores = load_scores(run_name) or {}
    fid = scores.get("fid_2048", (float("nan"), float("nan")))
    inception = scores.get("inception_score", (float("nan"), float("nan")))
    sigma = "calib" if hp["likelihood"] == "gaussian_calibrated" else str(hp["gaussian_sigma"])
    print(
        f"{name:14s} {hp['beta']:5.2f} {sigma:>6s} {run_i['best_epoch']:5d} {best['kl']:7.1f} "
        f"{m['val_mse']:8.5f} {m['hf_mse']:8.5f} {m['active_dims']:4d} "
        f"{fid[0]:7.1f} +- {fid[1]:4.1f} {inception[0]:6.2f} +- {inception[1]:4.2f}"
    )
    points.append((name, best["kl"], m["val_mse"], fid[0]))

plt.figure(figsize=(5, 3.5))
for name, kl, mse, fid in points:
    plt.scatter(kl, mse, color="C0")
    plt.annotate(f"{name[2:]}\nFID {fid:.0f}", (kl, mse), fontsize=7, xytext=(4, 4), textcoords="offset points")
for label, mse in pca_lines.items():
    plt.axhline(mse, color="gray", linestyle="--", linewidth=0.8)
    plt.text(0.01, mse, label, fontsize=7, va="bottom", color="gray", transform=plt.gca().get_yaxis_transform())
plt.xlabel("rate: KL [nats per image, validation]")
plt.ylabel("distortion: val MSE")
plt.tight_layout()
plt.savefig(FIGURES_DIR / f"rate_distortion_{datetime.now():%m%d-%H%M%S}.pdf")
plt.show()

In [ ]:
@torch.no_grad()
def aggregate_posterior_sampler(model):
    # Plumbing: one Gaussian fitted to the aggregate posterior q(z) = mean over the training images i of
    # q(z | x_i) (moment matching: mean of the mu_i; covariance of the mu_i plus the mean posterior variance).
    # Sampling z from it instead of from the prior N(0, I) avoids regions of the latent space that no
    # training image is encoded to (Dai & Wipf 2019, "Diagnosing and enhancing VAE models").
    mu, logvar, _, _ = reconstruct_all(model, train)
    covariance = torch.cov(mu.T) + torch.diag(logvar.exp().mean(0)) + 1e-6 * torch.eye(mu.shape[1], device=device)
    fitted = torch.distributions.MultivariateNormal(mu.mean(0), covariance_matrix=covariance)
    return lambda n: torch.sigmoid(model.decode(fitted.sample((n,))))


def score_sampler(sampler, repeats=1 if QUICK else 3):
    # Like fid_and_is (section 7), for any function n -> n images
    runs = [scorer(sampler(len(test_images)).cpu(), test_images) for _ in range(repeats)]
    return {
        name: (np.mean([r[name] for r in runs]), np.std([r[name] for r in runs]))
        for name in runs[0]
    }


print(f"{'name':14s} {'FID prior':>10s} {'FID aggregate':>14s} {'IS prior':>9s} {'IS aggregate':>13s}")
best_fid, best_name, best_model = float("inf"), None, None
for name in B:
    run_name = newest_run(name)
    scores_file = MODELS_DIR / f"{run_name}_scores.json" if run_name else None
    if scores_file is None or not scores_file.exists():
        print(f"{name:14s} (not scored yet: run the first cell of this section)")
        continue
    model_i, run_i = load_vae(run_name)
    with open(scores_file) as f:
        saved = json.load(f)
    if "samples_aggregate" not in saved or RESCORE:
        saved["samples_aggregate"] = score_sampler(aggregate_posterior_sampler(model_i))
        with open(scores_file, "w") as f:
            json.dump(saved, f, indent=2)
    prior, aggregate = saved["samples"], saved["samples_aggregate"]
    print(
        f"{name:14s} {prior['fid_2048'][0]:10.1f} {aggregate['fid_2048'][0]:14.1f} "
        f"{prior['inception_score'][0]:9.2f} {aggregate['inception_score'][0]:13.2f}"
    )
    if prior["fid_2048"][0] < best_fid:
        best_fid, best_name, best_model = prior["fid_2048"][0], name, model_i

if best_model is not None:  # the same 16 draws from the prior (top) and from the fitted Gaussian (bottom)
    with torch.random.fork_rng(devices=[device] if device.type == "cuda" else []):
        torch.manual_seed(0)
        from_prior = best_model.sample(16)
        torch.manual_seed(0)
        from_aggregate = aggregate_posterior_sampler(best_model)(16)
    panel = torch.cat([from_prior, from_aggregate]).cpu()
    show(panel, f"{best_name}: z ~ N(0, I) (rows 1-2) vs z ~ fitted aggregate posterior (rows 3-4)")
    save_image(panel, FIGURES_DIR / f"samples_prior_vs_aggregate_{datetime.now():%m%d-%H%M%S}.png", nrow=8, pad_value=1.0)

## 14. Final: the regularized stack with seeds, scored from the prior and from the fitted prior

Sections 11 to 13 decided one thing each. This section puts them together once, with error bars, and is the
model of the report: the architecture winners of section 11 (`a-stack`) plus the regularization winners of
section 12 (`augment`, `dropout = 0.2`, i.e. `r-stack`), at the two KL weights worth reporting from section 13:
`beta = 0.1` (the reconstruction setting) and `beta = 1` (the best FID of samples from the prior). Three seeds per
`beta`, so every number in the table is a mean +- std over runs. Each run is scored twice, like section 13:
samples from the prior $N(0, I)$ (the handout's metric) and samples from the Gaussian fitted to the aggregate
posterior.

Run names are `final-beta<beta>-seed<seed>-<stamp>`; nothing from sections 1 to 13 is reused or changed, and
`train_or_load` skips runs that exist. Needs the helpers of sections 7 (functions, the result cell with `plot_history`, the FID cell),
9 (`compare_figure`), 10 (`pca_reconstruct`) and 13 (`load_scores`, the samplers): on a fresh kernel run those cells first, or Run All.
About 45 minutes from scratch. The same setting as a CLI config for the README pipeline (`hw1.train`,
`hw1.evaluate`, `hw1.report_assets`, which fill `report/generated/`): `configs/final.yaml`.

In [ ]:
FINAL_BASE = {**A_STACK, "augment": True, "dropout": 0.2}  # = r-stack: section 11 + section 12 winners
FINAL_BETAS = [0.1, 1.0]  # section 13: beta 0.1 = the reconstruction setting, beta 1 = best prior-sample FID
FINAL_SEEDS = [0, 1, 2]  # three runs per beta -> mean +- std in the table

final_runs = {}  # (beta, seed) -> (model, run)
for beta in FINAL_BETAS:
    for seed in FINAL_SEEDS:
        settings = {**FINAL_BASE, "beta": beta, "seed": seed}
        final_runs[(beta, seed)] = train_or_load(f"final-beta{beta:g}-seed{seed}", settings)

In [ ]:
FINAL_RESCORE = False  # True: compute the scores again even when a `_scores.json` exists

for model_i, run_i in final_runs.values():
    scores_file = MODELS_DIR / f"{run_i['name']}_scores.json"
    saved = json.loads(scores_file.read_text()) if scores_file.exists() else {}
    if "samples" not in saved or FINAL_RESCORE:  # prior samples, 3 repeats (fid_and_is, section 7)
        saved["samples"], saved["real_images"] = fid_and_is(model_i), real_scores
    if "samples_aggregate" not in saved or FINAL_RESCORE:  # fitted aggregate posterior (section 13)
        saved["samples_aggregate"] = score_sampler(aggregate_posterior_sampler(model_i))
    with open(scores_file, "w") as f:
        json.dump(saved, f, indent=2)
    print(
        f"{run_i['name']:30s} FID-2048 prior {saved['samples']['fid_2048'][0]:6.1f}   "
        f"fitted prior {saved['samples_aggregate']['fid_2048'][0]:6.1f}"
    )

In [ ]:
columns = ["val_mse", "hf_mse", "gap", "kl", "best_epoch", "fid_prior", "is_prior", "fid_aggregate", "is_aggregate"]
final_rows = []
for (beta, seed), (model_i, run_i) in final_runs.items():
    m, best = run_i["metrics"], run_i["history"]["val"][run_i["best_epoch"] - 1]
    scores = json.loads((MODELS_DIR / f"{run_i['name']}_scores.json").read_text())
    final_rows.append(
        {
            "run": run_i["name"],
            "beta": beta,
            "seed": seed,
            "params": count_parameters(model_i),
            "val_mse": m["val_mse"],
            "hf_mse": m["hf_mse"],
            "gap": m["val_mse"] / m["train_mse"] - 1,
            "kl": best["kl"],
            "best_epoch": run_i["best_epoch"],
            "fid_prior": scores["samples"]["fid_2048"][0],
            "is_prior": scores["samples"]["inception_score"][0],
            "fid_aggregate": scores["samples_aggregate"]["fid_2048"][0],
            "is_aggregate": scores["samples_aggregate"]["inception_score"][0],
            "train_time_s": run_i.get("train_time_s", float("nan")),
        }
    )

print(
    f"reference: real train images vs test  FID-2048 {real_scores['fid_2048']:.1f}  "
    f"IS {real_scores['inception_score']:.2f};  PCA K=128 val_mse 0.00188"
)
print(f"{'beta':>5s} {'runs':>4s} " + " ".join(f"{c:>18s}" for c in columns))
for beta in FINAL_BETAS:
    group = [row for row in final_rows if row["beta"] == beta]
    cells = [f"{np.mean([r[c] for r in group]):.4g} +- {np.std([r[c] for r in group]):.2g}" for c in columns]
    print(f"{beta:5.2f} {len(group):4d} " + " ".join(f"{cell:>18s}" for cell in cells))

with open(FIGURES_DIR / f"final_table_{datetime.now():%m%d-%H%M%S}.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(final_rows[0]))
    writer.writeheader()
    writer.writerows(final_rows)

# the reported model: the beta with the better mean FID of prior samples (the handout's metric), seed 0
FINAL_BETA = min(
    FINAL_BETAS, key=lambda b: np.mean([r["fid_prior"] for r in final_rows if r["beta"] == b])
)
final_model, final_run = final_runs[(FINAL_BETA, FINAL_SEEDS[0])]
print(f"\nreported run: {final_run['name']}  (beta {FINAL_BETA:g}; configs/final.yaml must say the same)")

# figures for the report: loss curves, reconstructions next to PCA and the hand-written model, samples both ways
plot_history(final_run["history"], final_run["name"])
base_model, base_run = load_vae("vae-base")
pca_128 = pca_reconstruct(x_fixed.flatten(1).to(device), 128).view(-1, 1, SIZE, SIZE).cpu()
compare_figure(
    [
        ("validation\nimages", x_fixed),
        ("PCA\nK=128", pca_128),
        (label_for("vae-base", base_run), base_model.reconstruct(x_fixed.to(device)).cpu()),
        (label_for("final", final_run), final_model.reconstruct(x_fixed.to(device)).cpu()),
    ],
    "reconstructions through the posterior mean",
    "final_reconstructions",
)
with torch.random.fork_rng(devices=[device] if device.type == "cuda" else []), torch.no_grad():
    torch.manual_seed(0)
    sample_rows = [
        (label_for("vae-base", base_run), base_model.sample(8).cpu()),
        (label_for("final", final_run) + "\nz ~ N(0, I)", final_model.sample(8).cpu()),
        (label_for("final", final_run) + "\nz ~ fitted q(z)", aggregate_posterior_sampler(final_model)(8).cpu()),
    ]
compare_figure(sample_rows, "new images, all 8 draws", "final_samples")

## 15. Into the report

- Figures: `report/figures/<run>_loss_curves.pdf`, `<run>_samples.png`, `<run>_reconstructions.png`, `grid_search.pdf`,
  the comparisons `compare_reconstructions_<stamp>.pdf` / `compare_samples_<stamp>.pdf` (section 9), the
  rate-distortion curve `rate_distortion_<stamp>.pdf` and `samples_prior_vs_aggregate_<stamp>.png` (section 13),
  and from section 14 `<final run>_loss_curves.pdf`, `final_reconstructions_<stamp>.pdf`, `final_samples_<stamp>.pdf`.
- Numbers: the FID / IS printout of section 7 (`models/<name>_scores.json`), the hyperparameters in `hparams`,
  the grid table (8), the summary table (9), the results table (10, `report/figures/results_table_<stamp>.csv`),
  the one-change tables (11, 12), the rate-distortion table (13), and the seeds table
  (14, `report/figures/final_table_<stamp>.csv`: mean +- std over seeds, prior and fitted-prior FID).
- The reported run itself goes through the README pipeline so that `report/generated/` is filled automatically:
  `configs/final.yaml` (same setting as section 14) -> `hw1.train` -> `hw1.evaluate` -> `hw1.report_assets`.
- Build: `uv run python scripts/build_latex.py homeworks/hw1/report`.
